# [STARTER] Exercise - Create an Agent with external API call enabled

In this exercise, you'll build an agent that can interact with external APIs to gather real-time data
and provide responses based on that information. You'll combine concepts from state management and
memory while adding the ability to make external API calls safely and effectively.


## Challenge

Your task is to create an agent that can make External API Calls:

- Implement tools that interact with real APIs
- Handle API responses and errors gracefully
- Use environment variables for API keys
- Process and format API data for user consumption

## Setup
First, let's import the necessary libraries:

In [1]:
import os
import random
from typing import List
import requests
from dotenv import load_dotenv

from lib.agents import Agent
from lib.messages import BaseMessage
from lib.tooling import tool

In [2]:
load_dotenv()

True

## Define API tools

Feel free to use any open service available through APIs.

Here are a few examples. You can follow the instructions given.
- https://jsonplaceholder.typicode.com/guide/
- https://www.exchangerate-api.com/
- https://openweathermap.org/

Or you can find one you're interested in here.
- https://github.com/public-apis/public-apis

In [ ]:
# TODO: Define as many tools that access external APIs as you want
# Example:
@tool
def get_random_pokemon() -> dict:
    """Get a random Pokemon from the original 151"""
    URL = "https://pokeapi.co/api/v2/pokemon?limit=151"
    response = requests.get(URL)
    response.raise_for_status()
    return random.choice(response.json()['results'])

@tool
def get_weather_now(city: str) -> dict:
    """Get the current weather for a given city using OpenWeatherMap API"""
    API_KEY = os.getenv("OPEN_WEATHER_API_KEY")    
    BASE_URL = "https://api.openweathermap.org/data/2.5/weather"


    URL = f"{BASE_URL}?appid={API_KEY}&q={city}&units=metric"

    response = requests.get(URL)
    response.raise_for_status()
    return response.json()

In [4]:
# TODO: Add all the tools you have defined
tools = [get_random_pokemon, get_weather_now]

In [5]:
# TODO: Add instructions to your agent

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=(
        """You are an assistant that can help with:
        
        - Get a random Pokemon using the get_random_pokemon tool.
        - Get the current weather for a given city (in English) using the get_weather_now tool.
        """
    ),
    tools=tools
)

In [6]:
def print_messages(messages: List[BaseMessage]):
    for m in messages:
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

## Run your Agent

In [7]:
# TODO: Change the query and then run your agent
query = "Tell me a story about a random Pokemon who lives in São Paulo today."
session_id = "external_tools"

In [8]:
run1 = agent.invoke(
    query=query, 
    session_id=session_id,
)

print("\nMessages from run 1:")
messages = run1.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 1:
 -> (role = system, content = You are an assistant that can help with:

        - Get a random Pokemon using the get_random_pokemon tool.
        - Get the current weather for a given city (in English) using the get_weather_now tool.
        , tool_calls = None)
 -> (role = user, content = Tell me a story about a random Pokemon who lives in São Paulo today., tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [ChatCompletionMessageToolCall(id='call_k74jm4QrjAJgL6FdwYROwKXE', function=Function(arguments='{}', name='get_random_pokemon'), type='function')])
 -> (role = tool, content = "{'name': 'marowak', '

## Check session histories

In [9]:
runs = agent.get_session_runs(session_id)
for i, run_object in enumerate(runs, 1):
    print(f"\n# Run {i}", run_object.metadata)
    print("Messages:")
    print_messages(run_object.get_final_state()["messages"])


# Run 1 {'run_id': '48bc9da4-b379-4f81-af6f-a300d781f9a6', 'start_timestamp': '2026-03-09 17:53:11.892658', 'end_timestamp': '2026-03-09 17:53:27.570953', 'snapshot_counts': 7}
Messages:
 -> (role = system, content = You are an assistant that can help with:

        - Get a random Pokemon using the get_random_pokemon tool.
        - Get the current weather for a given city (in English) using the get_weather_now tool.
        , tool_calls = None)
 -> (role = user, content = Tell me a story about a random Pokemon who lives in São Paulo today., tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [ChatCompletionMessageToolCall(id='call_k74jm4QrjAJgL6FdwYROwKXE', function=Function(arguments='{}', name='get_random_pokemon'), type='function')])
 -> (role = tool, content = "{'name': 'marowak', 'url': 'https://pokeapi.co/api/v2/pokemon/105/'}", tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [ChatCompletionMessageToolCall(id='call_ZVvVC1SacNroJAuFdcnG7

In [10]:
runs = agent.get_session_runs(session_id)
for run_object in runs:
    print(run_object)
    for snp in run_object.snapshots:
        print(f"-> {snp}")
    print("\n")

Run('48bc9da4-b379-4f81-af6f-a300d781f9a6')
-> Snapshot('3191e5eb-dab5-4c73-af1d-f2ef1b45bb28') @ [2026-03-09 17:53:11.892842]: __entry__.State({'user_query': 'Tell me a story about a random Pokemon who lives in São Paulo today.', 'instructions': 'You are an assistant that can help with:\n\n        - Get a random Pokemon using the get_random_pokemon tool.\n        - Get the current weather for a given city (in English) using the get_weather_now tool.\n        ', 'messages': [], 'current_tool_calls': None, 'session_id': 'external_tools'})
-> Snapshot('ad899756-4f97-4690-addb-d529e7a37f4f') @ [2026-03-09 17:53:11.893049]: message_prep.State({'user_query': 'Tell me a story about a random Pokemon who lives in São Paulo today.', 'instructions': 'You are an assistant that can help with:\n\n        - Get a random Pokemon using the get_random_pokemon tool.\n        - Get the current weather for a given city (in English) using the get_weather_now tool.\n        ', 'messages': [SystemMessage(con